<a href="https://colab.research.google.com/github/AliyaBadmaeva/PDP/blob/main/Badmaeva_AA_RuBert_PDP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datetime import datetime
current_datetime = datetime.now()
print(current_datetime)

2025-09-10 04:53:35.618647


In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import pandas as pd
from sklearn.model_selection import train_test_split

In [4]:
from tqdm import tqdm

In [5]:
#!pip uninstall transformers

In [6]:
#!pip install transformers

In [7]:
!pip show transformers

Name: transformers
Version: 4.52.4
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /home/aliuska/anaconda3/envs/my_pdp/lib/python3.13/site-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: 


In [8]:
import warnings
# игнорируем предупреждения
warnings.filterwarnings('ignore')

In [9]:
#!pip install openpyxl  # установка бибилиотеки для чтения Excel

In [10]:
DATASET = 'stepik_all_reviews_2025-09-07_05-10-59.xlsx'

In [11]:
df = pd.read_excel(DATASET) # читаем датасет из отзывов со Степика

In [12]:
df.head(10)  # Первые 10 строк датасета

,Курс,ID отзыва,Звёзды,Отзыв
0,240918,446792,4,"Спсибо за курс, очень огромный труд и силы вло..."
1,240918,443604,5,NaN
2,1612,353623,2,Никогда в жизни я не стал бы проходить этот ку...
3,1612,286194,5,"Сложный, муторный, но как по мне очень хороший..."
4,1612,277139,4,"Начало курса очень хорошее, но к концу всё ста..."
5,1612,234362,3,"Курс безусловно полезен, но уход в анализ гене..."
6,1612,218904,3,на 22 год некоторые аспекты данного курса не а...
7,1612,210127,1,Про docker 30% курса
8,1612,189023,5,Отличный курс
9,1612,187622,3,"К сожалению мои ожидания не совпали, с тем что..."


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9580 entries, 0 to 9579
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Курс       9580 non-null   int64 
 1   ID отзыва  9580 non-null   int64 
 2   Звёзды     9580 non-null   int64 
 3   Отзыв      9579 non-null   object
dtypes: int64(3), object(1)
memory usage: 299.5+ KB


In [14]:
df.describe()  # статистические данные

,Курс,ID отзыва,Звёзды
count,9580.000000,9580.000000,9580.000000
mean,90040.079123,291593.991962,3.418789
std,78642.859329,121485.961652,1.533555
min,7.000000,31.000000,0.000000
25%,191.000000,207628.250000,2.000000
50%,98403.000000,318745.000000,3.000000
75%,171826.000000,386886.500000,5.000000
max,251270.000000,459842.000000,5.000000


In [15]:
df.rename(columns={'Звёзды': 'label', 'Отзыв' :'text'}, inplace=True)  # заменяем названия на кириллице

In [16]:
del df['Курс']  # удаляем лишние признаки

In [17]:
del df['ID отзыва']  # удаляем ID отзыва

In [18]:
df['label'].value_counts()  # количество данных в целевом признаке

label
5    3890
3    2306
1    1752
2     946
4     685
0       1
Name: count, dtype: int64

In [19]:
df[df['label']==0]  # отзывы, имеющие 0 баллов

,label,text
290,0,[текст не найден]


Отзыв в ячейке выше не удалось спарсить, а пустые отзывы не несут никакой пользы, удалим их.

In [20]:
df = df[df['label'] != 0]  # перезапишем датасет, исключим отзывы, имеющие 0 баллов по фильтру

In [21]:
df['label'].value_counts()

label
5    3890
3    2306
1    1752
2     946
4     685
Name: count, dtype: int64

Преобразуем значения в 3 тональности: положительная, нейтральная и отрицательная, где 1,2 балла - отрицательная, 3 балла - нейтральная, а 4 и 5 баллов - положительная тональность отзывов.

In [22]:
df["label"] = df["label"].map({1:0, 2:0, 3:1, 4:2, 5:2})  # три тональности

In [23]:
df['label'].value_counts()

label
2    4575
0    2698
1    2306
Name: count, dtype: int64

In [24]:
# удалим нулевые значения по наличию в столбце отзыв, чтобы не было потом проблем с созданием тензоров
df.dropna(axis=0, how='any',subset=['text'], inplace=True)

In [25]:
df.duplicated().sum()  # количество дублей

np.int64(0)

In [26]:
df = df.drop_duplicates(subset=['text'])  # удалим дибли по столбцу с текстом отзыва

In [27]:
df.describe()

,label
count,9578.000000
mean,1.195866
std,0.849089
min,0.000000
25%,0.000000
50%,1.000000
75%,2.000000
max,2.000000


In [28]:
df = df[df['text'].str.count(r'[А-Яа-я]') >= 2].reset_index(drop=True)  # Оставим только отзывы, имеющие кириллицу и по длине от 2 букв

In [29]:
def clean_text(t):  # Очистим текст
    t = str(t).strip()
    if len(t) < 10:           # короткие отзывы неинформативны
        return ""
    if t.lower() in {"ок", "none", "-", "+"}:
        return ""
    return t

df["text"] = df["text"].apply(clean_text)
df = df[df["text"] != ""].reset_index(drop=True)

In [30]:
print(df.shape)          # сколько осталось (строки, столбцы)
print(df['text'].head()) # примеры

(9119, 2)
0    Спсибо за курс, очень огромный труд и силы вло...
1    Никогда в жизни я не стал бы проходить этот ку...
2    Сложный, муторный, но как по мне очень хороший...
3    Начало курса очень хорошее, но к концу всё ста...
4    Курс безусловно полезен, но уход в анализ гене...
Name: text, dtype: object


In [31]:
def balanced_dataset_len(df: pd.DataFrame,
                         text_col: str = 'text',
                         label_col: str = 'label',
                         n_per_class: int = 2000,
                         seed: int = 42) -> pd.DataFrame:
    """
    Балансировка по label и равномерное распределение по длине текста
    (квантили 0-33 %, 33-66 %, 66-100 %)
    """
    df = df.copy()
    df['length'] = df[text_col].str.len()

    def sample_quantiles(x):
        if len(x) <= n_per_class // 3:          # мало данных – берём всё
            return x
        # делим на 3 квантиля по длине
        x = x.sort_values('length')
        q1, q2 = x['length'].quantile([0.33, 0.66])
        parts = [x[x['length'] <= q1],
                 x[(x['length'] > q1) & (x['length'] <= q2)],
                 x[x['length'] > q2]]
        # из каждого квантиля берём поровну
        n_from_part = n_per_class // 3
        return pd.concat([p.sample(min(len(p), n_from_part), random_state=seed)
                          for p in parts], ignore_index=True)

    out = (df.groupby(label_col, group_keys=False)
             .apply(sample_quantiles)
             .reset_index(drop=True))

    # если вдруг набралось чуть больше – обрезаем до ровного числа
    out = (out.groupby(label_col, group_keys=False)
             .apply(lambda x: x.sample(min(len(x), n_per_class), random_state=seed))
             .reset_index(drop=True))

    return out.drop(columns='length')


# использование
df_bal = balanced_dataset_len(df, text_col='text', label_col='label', n_per_class=2000)
print(df_bal['label'].value_counts())
print(df_bal['text'].str.len().describe())   # проверим распределение длин

label
0    1998
1    1998
2    1998
Name: count, dtype: int64
count    5994.000000
mean      245.310811
std       387.818104
min        10.000000
25%        48.000000
50%       115.000000
75%       271.000000
max      7201.000000
Name: text, dtype: float64


In [33]:
train_df, val_df = train_test_split(
    df,
    test_size=0.15,          # 15 % на валидацию
    stratify=df['label'],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print('Train:', train_df['label'].value_counts())
print('Val  :', val_df['label'].value_counts())

Train: label
2    3668
0    2199
1    1884
Name: count, dtype: int64
Val  : label
2    648
0    388
1    332
Name: count, dtype: int64


In [34]:
X_train = train_df.text
X_val = val_df.text

y_train = train_df.label
y_val = val_df.label

In [35]:
! nvidia-smi

Wed Sep 10 04:53:40 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.51.02              Driver Version: 576.02         CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3070 ...    On  |   00000000:01:00.0 Off |                  N/A |
| N/A   30C    P8             13W /   82W |     123MiB /   8192MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [36]:
MAX_LEN = 256

In [37]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [38]:
#!pip install torch

In [39]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

In [40]:
base = "DeepPavlov/rubert-base-cased"   # без головы
tokenizer = AutoTokenizer.from_pretrained(base)


In [41]:
tokenizer.save_pretrained('.')

('./tokenizer_config.json',
 './special_tokens_map.json',
 './vocab.txt',
 './added_tokens.json',
 './tokenizer.json')

In [42]:
from transformers import DataCollatorWithPadding
from datasets import Dataset

In [43]:
def tok_func(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

train_ds = train_df[["text", "label"]].rename(columns={"label": "labels"})
val_ds = val_df[["text", "label"]].rename(columns={"label": "labels"})

train_ds = Dataset.from_pandas(train_ds)
val_ds = Dataset.from_pandas(val_ds)

# 2. Токенизация на лету + dynamic padding
train_ds = train_ds.map(tok_func, batched=True)
val_ds = val_ds.map(tok_func, batched=True)

# 3. DataCollator делает паддинг «по максимуму внутри батча»
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/7751 [00:00<?, ? examples/s]

Map:   0%|          | 0/1368 [00:00<?, ? examples/s]

In [44]:
#!pip install --upgrade nbformat  # из-за проблем отображения на Github
#!pip install --upgrade nbconvert

In [45]:
from transformers import Trainer, TrainingArguments

In [46]:
#!pip install evaluate

In [47]:
id2label = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"}
label2id = {"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2}

In [48]:
model = AutoModelForSequenceClassification.from_pretrained(
            base,
            num_labels=3,          # 3 тональности
            problem_type="single_label_classification",
            id2label=id2label,
            label2id=label2id).to("cuda")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [49]:
from torch.optim import AdamW

loss_fn = torch.nn.CrossEntropyLoss()

In [50]:
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
    }

In [51]:
from transformers import TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./ruBERT_sentiment",
    overwrite_output_dir=True,
    num_train_epochs=8,                 # 8 эпох
    per_device_train_batch_size=4,      #  batch на GPU
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=8,      # эффективный batch = 8×4 = 32
    learning_rate=1e-5,                 # lr для AdamW
    weight_decay=0.02,                  # L2-регуляризация
    warmup_ratio=0.15,                   # 10 % шагов – линейный разогрев
    lr_scheduler_type="cosine",         # косинусное затухание lr
    bf16=False,                         # у RTX 3070 нет bf16, поэтому
    fp16=True,                          # экономия 30 % памяти + скорость
    dataloader_num_workers=4,           # быстрее загрузка
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="accuracy",         # следим за accuracy
    load_best_model_at_end=True,
    report_to="none",                   # не шлём логи в wandb
    greater_is_better=True,
    save_total_limit=2,                 # храним 2 лучших чек-поинта
)
model.gradient_checkpointing_enable()

In [52]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [53]:
# train the model
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.687689,0.690058
2,No log,0.660197,0.716374
3,0.759600,0.640893,0.728801
4,0.759600,0.673192,0.735380
5,0.437900,0.741485,0.725146
6,0.437900,0.811549,0.725877
7,0.259300,0.837932,0.728070


TrainOutput(global_step=1701, training_loss=0.4500902659188012, metrics={'train_runtime': 1028.9789, 'train_samples_per_second': 60.262, 'train_steps_per_second': 1.889, 'total_flos': 2909065197130596.0, 'train_loss': 0.4500902659188012, 'epoch': 7.0})

In [54]:
# оценка текущей модели после обучения
trainer.evaluate()

{'eval_loss': 0.6731916069984436,
 'eval_accuracy': 0.7353801169590644,
 'eval_runtime': 3.6172,
 'eval_samples_per_second': 378.19,
 'eval_steps_per_second': 11.888,
 'epoch': 7.0}

In [55]:
# !pip install 'accelerate>=0.26.0'

In [56]:
from transformers import pipeline

In [57]:
trainer.save_model(output_dir='./ruBert_results')

In [58]:
text = """
Мне не нравится эта дисциплина. Мне больше по душе такие дисциплины, как МО, например.
"""


In [59]:
classifier = pipeline("sentiment-analysis", model="./ruBert_results")
classifier(text)

Device set to use cuda:0


[{'label': 'NEGATIVE', 'score': 0.8201292157173157}]

In [60]:
current_datetime = datetime.now()
print(current_datetime)

2025-09-10 05:11:11.042215


In [61]:
# из-за проблем отображения на Github приходится вручную удалять виджеты
import json, io, os, glob

# 1. находим .ipynb в текущей папке
files = glob.glob('*.ipynb')
if not files:
    raise RuntimeError('Нет .ipynb в папке')
nb = files[0]                                   # берём первый

# 2. читаем JSON
with io.open(nb, 'r', encoding='utf-8') as f:
    data = json.load(f)

# 3. УДАЛЯЕМ widgets везде
for cell in data.get('cells', []):
    cell.get('metadata', {}).pop('widgets', None)

# 4. перезаписываем
with io.open(nb, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=1, ensure_ascii=False)

print('widgets удалены')

widgets удалены


In [62]:
!jupyter nbconvert "$nb" --to html

[NbConvertApp] Converting notebook Untitled.ipynb to html
[NbConvertApp] Writing 296071 bytes to Untitled.html
